In [1]:
from __future__ import annotations

from configparser import ConfigParser
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import tifffile

In [2]:
@dataclass
class CTScanData:
    sample_dir: Path
    settings_path: Path
    projections_dir: Path
    settings: dict[str, dict[str, Any]]
    projections: np.ndarray
    projection_files: list[Path]

In [3]:
class CasePreservingConfigParser(ConfigParser):
    def optionxform(self, optionstr: str) -> str:
        return optionstr

In [4]:
def _parse_value(raw: str) -> Any:
    value = raw.strip()
    upper = value.upper()

    if upper == "TRUE":
        return True
    if upper == "FALSE":
        return False

    try:
        if any(ch in value for ch in (".", "e", "E")):
            return float(value)
        return int(value)
    except ValueError:
        return value

In [5]:
def load_cto_settings(settings_path: str | Path) -> dict[str, dict[str, Any]]:
    settings_path = Path(settings_path)
    parser = CasePreservingConfigParser()

    with settings_path.open("r", encoding="utf-8") as handle:
        parser.read_file(handle)

    parsed: dict[str, dict[str, Any]] = {}
    for section in parser.sections():
        parsed[section] = {
            key: _parse_value(value)
            for key, value in parser.items(section)
        }
    return parsed

In [6]:
def list_projection_files(projections_dir: str | Path) -> list[Path]:
    projections_dir = Path(projections_dir)
    files = sorted(projections_dir.glob("*.tif"))
    if not files:
        raise FileNotFoundError(f"No TIFF projections found in {projections_dir}")
    return files

In [7]:
def load_projection_stack(projections_dir: str | Path, dtype=np.float32) -> tuple[np.ndarray, list[Path]]:
    files = list_projection_files(projections_dir)
    stack = np.stack(
        [tifffile.imread(str(path)).astype(dtype, copy=False) for path in files],
        axis=0,
    )
    return stack, files

In [8]:
def load_sample1(sample_dir: str | Path = "sample 1", dtype=np.float32) -> CTScanData:
    sample_dir = Path(sample_dir)
    settings_path = sample_dir / "settings.cto"
    projections_dir = sample_dir / "projections"

    if not settings_path.exists():
        raise FileNotFoundError(f"Missing settings file: {settings_path}")
    if not projections_dir.exists():
        raise FileNotFoundError(f"Missing projections directory: {projections_dir}")

    settings = load_cto_settings(settings_path)
    projections, projection_files = load_projection_stack(projections_dir, dtype=dtype)

    return CTScanData(
        sample_dir=sample_dir,
        settings_path=settings_path,
        projections_dir=projections_dir,
        settings=settings,
        projections=projections,
        projection_files=projection_files,
    )

In [9]:
if __name__ == "__main__":
    data = load_sample1()
    print("Loaded sample:", data.sample_dir)
    print("Projection stack shape:", data.projections.shape)
    print("Projection dtype:", data.projections.dtype)
    print("Number of projection files:", len(data.projection_files))
    print("Sections:", list(data.settings))


Loaded sample: sample 1
Projection stack shape: (359, 1000, 1000)
Projection dtype: float32
Number of projection files: 359
Sections: ['Device settings', 'Detector settings', 'CT scan settings', 'CT reconstruction settings']


Data Loading Is done !!

In [10]:
from __future__ import annotations

from dataclasses import dataclass, replace
from pathlib import Path
from typing import Any

import numpy as np

from data_loader import load_cto_settings

In [11]:
@dataclass
class CTGeometry:
    settings_path: Path
    projections: int
    angle_range_deg: float
    angles_deg: np.ndarray
    angles_rad: np.ndarray
    detector_rows: int
    detector_cols: int
    detector_pixel_size_mm: float
    source_to_object_mm: float
    source_to_detector_mm: float
    center_of_rotation_px: float
    vertical_center_px: float
    recon_rows: int
    recon_cols: int
    zmin: int
    zmax: int
    direction: int

In [12]:
def _require(section: dict[str, Any], key: str) -> Any:
    if key not in section:
        raise KeyError(f"Missing required key '{key}'")
    return section[key]

In [13]:
def parse_geometry(settings_path: str | Path) -> CTGeometry:
    settings_path = Path(settings_path)
    settings = load_cto_settings(settings_path)

    detector = settings["Detector settings"]
    scan = settings["CT scan settings"]
    recon = settings["CT reconstruction settings"]

    projections = int(_require(scan, "projections"))
    angle_range_deg = float(_require(scan, "angle range"))
    direction = int(_require(recon, "direction"))

    start_angle_deg = 0.0
    stop_angle_deg = start_angle_deg + direction * angle_range_deg
    angles_deg = np.linspace(start_angle_deg, stop_angle_deg, projections, endpoint=False, dtype=np.float32)
    angles_rad = np.deg2rad(angles_deg).astype(np.float32)

    detector_rows = int(_require(recon, "rows"))
    detector_cols = int(_require(recon, "columns"))
    detector_pixel_size_mm = float(_require(detector, "pixel size"))

    return CTGeometry(
        settings_path=settings_path,
        projections=projections,
        angle_range_deg=angle_range_deg,
        angles_deg=angles_deg,
        angles_rad=angles_rad,
        detector_rows=detector_rows,
        detector_cols=detector_cols,
        detector_pixel_size_mm=detector_pixel_size_mm,
        source_to_object_mm=float(_require(scan, "SOD")),
        source_to_detector_mm=float(_require(scan, "SDD")),
        center_of_rotation_px=float(_require(recon, "COR")),
        vertical_center_px=float(_require(recon, "vertical center")),
        recon_rows=int(_require(recon, "rows")),
        recon_cols=int(_require(recon, "columns")),
        zmin=int(_require(recon, "zmin")),
        zmax=int(_require(recon, "zmax")),
        direction=direction,
    )

In [14]:
def geometry_for_projection_count(geometry: CTGeometry, projection_count: int) -> CTGeometry:
    if projection_count <= 0:
        raise ValueError("projection_count must be positive")

    start_angle_deg = 0.0
    stop_angle_deg = start_angle_deg + geometry.direction * geometry.angle_range_deg
    angles_deg = np.linspace(
        start_angle_deg,
        stop_angle_deg,
        projection_count,
        endpoint=False,
        dtype=np.float32,
    )
    angles_rad = np.deg2rad(angles_deg).astype(np.float32)

    return replace(
        geometry,
        projections=projection_count,
        angles_deg=angles_deg,
        angles_rad=angles_rad,
    )

In [15]:
if __name__ == "__main__":
    geometry = parse_geometry("sample 1/settings.cto")
    print(geometry)

CTGeometry(settings_path=PosixPath('sample 1/settings.cto'), projections=360, angle_range_deg=360.0, angles_deg=array([  0.,   1.,   2.,   3.,   4.,   5.,   6.,   7.,   8.,   9.,  10.,
        11.,  12.,  13.,  14.,  15.,  16.,  17.,  18.,  19.,  20.,  21.,
        22.,  23.,  24.,  25.,  26.,  27.,  28.,  29.,  30.,  31.,  32.,
        33.,  34.,  35.,  36.,  37.,  38.,  39.,  40.,  41.,  42.,  43.,
        44.,  45.,  46.,  47.,  48.,  49.,  50.,  51.,  52.,  53.,  54.,
        55.,  56.,  57.,  58.,  59.,  60.,  61.,  62.,  63.,  64.,  65.,
        66.,  67.,  68.,  69.,  70.,  71.,  72.,  73.,  74.,  75.,  76.,
        77.,  78.,  79.,  80.,  81.,  82.,  83.,  84.,  85.,  86.,  87.,
        88.,  89.,  90.,  91.,  92.,  93.,  94.,  95.,  96.,  97.,  98.,
        99., 100., 101., 102., 103., 104., 105., 106., 107., 108., 109.,
       110., 111., 112., 113., 114., 115., 116., 117., 118., 119., 120.,
       121., 122., 123., 124., 125., 126., 127., 128., 129., 130., 131.,
       132.,